# STUDY NOTEBOOK 3: Layer 3, Estimation

**Who this is for:** a team member who needs to know Layer 3, the statistical heart of the project. You must be able to explain it to the guide, answer questions, and read `03_estimation.ipynb` and its helper files (`estimator.py`, `calls.py`, `variance.py`) without fear.

**What you need before you start:** basic Python and basic stats (average, probability, the bell curve). Everything else (bias, variance, weighting, clustering, prediction-powered inference, Monte Carlo) is taught here from zero, with toy code you can run.

**How to study this notebook**

1. Read top to bottom once and run every code cell (Shift+Enter).
2. After each section, cover the page and say the **Say it to the guide** box in your own words.
3. At the end: the **numbers to remember** table, the **glossary** and the **question list**. Mug those.
4. Then open the real `03_estimation.ipynb`. Section numbers here match its sections 1 to 10.

> **bold** = a term you must know. `code font` = a name that exists in the real code.

This is the hardest layer. Go slowly through sections 1 to 4. Everything after them is applications of those ideas.

## 0. What Layer 3 does, in plain words

**Input from Layer 2:** a calibrated probability for every pixel that it is flooded paddy.

**Problem:** we want, for each insurance **unit** (village or similar), **one number**: what share of its paddy area is flooded. Call it **theta_u** ("theta" is the Greek letter written as `theta`). And we want to know how sure we are of it, a **standard error s_u**.

**Two imperfect sources of information:**

| Source | Good | Bad |
|---|---|---|
| **The map** (Layer 2) | covers every plot, free | can be **biased**: it may systematically over- or under-read flooding, especially on a **new flood event** |
| **Field checks** (an officer walks to a plot and sees whether it is flooded) | trustworthy | **few** (we can only visit a handful) so they are noisy, and they may be taken from **convenient** places (near roads) so they can be biased too |

**The idea of Layer 3:** use **both**. Let the map give a guess for every plot, and use the few field checks only to **measure how wrong the map is** and correct it. Because the map's *error* varies much less than the flood itself, a few checks go a long way. This approach has a name: **prediction-powered inference (PPI)**.

**Outputs for every unit:**

1. **theta_u** with standard error **s_u**.
2. **The always-answered call:** `call_u = [theta_u > 0.25]` ("above" or "below"). **No abstain state.**
3. **P_u = Phi((theta_u - 0.25) / s_u)**, the probability that the unit is truly above the 25 percent line. A weak call shows up as P_u near 0.5, not as a refusal.
4. A **flip value** for farmer reports: how biased farmer reports would have to be before trusting them would flip the call.

**Project rules that apply here (quote them):**

* Hyperparameters are fitted on **training events only**; the validation region is used to look at methods; the **test region is scored once**, in a marked cell.
* Every method is compared with **baselines** (map only, checks only).
* We report **val-test gaps** and say when something is **simulated**. **The field checks here are simulated from hand labels**: no officer was observed.
* A **version history** is kept honestly (section 7).

## 1. Primer: the statistics words you need (read this twice)

### 1.1 Estimate, truth, error

* The **truth** of a unit is its real flooded share, written **theta_true**. In the real field we never see it. In our experiments we do, because the Sen1Floods11 hand labels give it. That is what lets us **score** the estimators.
* An **estimator** is a recipe that turns data into an estimate theta_hat of the truth.
* **Error** = estimate minus truth.

### 1.2 Bias and variance

Imagine repeating the whole field exercise many times with different random check draws.

* **Bias** = the average error over those repeats. An estimator is **unbiased** if its average error is 0, **biased** otherwise. Example: a map that always over-reads flooding by 8 percentage points has bias +0.08, no matter how many times you re-run it.
* **Variance** = how much the estimate jumps around from repeat to repeat. A small number of checks gives a high-variance estimate.
* **RMSE** (root mean squared error) = `sqrt(mean(error^2))`. It combines both. Lower is better. RMSE^2 = bias^2 + variance (roughly).
* **Standard error (s)** = the estimated size of the typical error. It is what we report as `s_u`.

Draw this: bias = how far the *centre* of the arrows is from the bullseye. Variance = how *spread out* the arrows are.

### 1.3 Confidence interval and coverage

A **90 percent interval** is `theta_hat +/- 1.645 * s`. (1.645 is the bell-curve number that leaves 5 percent in each tail.) If the standard errors are honest, the truth should fall inside the interval **90 percent of the time**. That fraction is the **coverage** (`cover90`). If coverage is, say, 70 percent, the standard errors are too small (over-confident). If it is 99 percent they are too big (timid).

### 1.4 Sampling words

* **Population / frame**: everything we could check. Here: every **plot** of paddy in a region.
* **Plot**: a small block of pixels (4 x 4 pixels at 20 m = **80 m**), the thing an officer checks. Its truth `y_c` is the fraction flooded (0 to 1; for a single check it is the flooded fraction).
* **Visit site**: a cluster of nearby plots (16 x 16 pixel block = **320 m**) that an officer checks from one stop (a hamlet, a stretch of embankment). In a visit, **k = 4 plots** are checked.
* **Unit**: village or similar; contains many sites, each with many plots.
* **Random design**: every plot has the same chance of being checked. **Route-style design**: plots near roads are checked far more often, because that is what officers can reach.
* **Inclusion probability pi_c** = the known chance that plot c was selected. (Recall notebook 1, section 5.) **Weight d_c = 1 / pi_c**.
* **Budget** = the average number of checked plots per unit. We test budgets **0, 4, 8, 16, 32, 64**.
* **Monte Carlo** = repeat the random experiment many times by computer (here 120 draws on validation, 200 on test) and average the results.

### 1.5 Strata and the "rectifier"

Plots are grouped (**stratified**) by their calibrated map score `f`:

* stratum **0 = mapped dry**: f < 0.15
* stratum **1 = uncertain**: 0.15 <= f < 0.6
* stratum **2 = mapped flooded**: f >= 0.6

These cut points (**0.15 and 0.6**) are **fixed in advance**, not tuned.

The **rectifier** of a plot is the map's error there: **e_c = y_c - f_c** (truth minus the map's probability). If the map said 0.8 and the plot was fully flooded (y = 1), e = +0.2. If the map said 0.8 and the plot was dry, e = -0.8. The checks exist to estimate the **average rectifier** in each stratum.

Why this works (the key insight): the flooded share varies hugely from plot to plot (0 or 1), but the **map's error** is a small, smoother quantity. Estimating a small, smooth quantity from a few checks is much easier than estimating the flooded share itself from those checks.

**Analogy:** you estimate the weight of 100 sacks by eye (the map). Weighing 5 sacks on a trusted scale tells you that your eye over-reads by 2 kg on average. You subtract 2 kg from all your eye estimates. Five weighings fix all 100 estimates, much better than weighing 5 and ignoring the rest.

**Say it to the guide (section 1):** "The map covers every plot but can be biased on a new event. Field checks are trustworthy but few and possibly convenient-biased. We use the map as a prediction for every plot and use the checks only to estimate the map's average error per stratum, the rectifier, and correct for it. That is prediction-powered inference."

## 2. The population frame and the unit formulas (section 1 of the notebook)

### 2.1 What the code builds (`Frame`, `frame_from_pixels`)

From per-pixel Layer 2 output, the code aggregates to **plots**: for each plot it stores its pixel count `w_c`, mean calibrated map probability `f_c`, true flooded fraction `y_c` (known only in experiments), unit, site, and (for the synthetic world) distance to a road.

For each unit u and stratum h:

* **W_uh** = the **share of the unit's pixels** that lie in stratum h (W_u0 + W_u1 + W_u2 = 1).
* **Fbar_uh** = the **mean map probability** over the unit's pixels in stratum h.
* **theta_map_u** = the map's raw mean for the unit = `sum_h W_uh * Fbar_uh` (exactly, when W is the pixel share and Fbar the pixel-weighted mean).
* **theta_true_u** = the true flooded share (experiments only).

### 2.2 The three regions in the real run

If Layer 2 artifacts exist the notebook uses them: **units = Layer 1 chip windows (pseudo-villages), plots = 4 x 4 pixel blocks, sites = 16 x 16 pixel blocks**, split **by event** (the Layer 1 split). Only **paddy pixels** (domain, paddy >= 0.5, labelled) count. A unit needs at least **1,000 paddy pixels** or it is dropped.

| Region | Events | Units in our run |
|---|---|---|
| **Train** | Mekong, Nigeria, Pakistan, Paraguay, Somalia, Spain, USA | **125** (7 events) |
| **Validation** | Sri-Lanka, Ghana | **27** |
| **Test** | India, Bolivia | **34** |

The real inputs come from **121 Layer 2 chip maps**. If the real files were missing, a documented **synthetic generator** (`synthetic_pixels`) would be used instead. It has a hierarchy unit to site to plot to pixel, unit-level and site-level map bias, and a **road effect** (plots near roads are less flooded but the map sees them as wetter), which is exactly what makes route-style checking biased.

### 2.3 The field checks are SIMULATED

In Layer 3 a "check" is simulated by reading the hand label `y_c` of the selected plot. **No officer was observed.** Say this every time.

Try it: build a small toy region and compute these quantities.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, math, json
from scipy.special import expit, ndtr
np.set_printoptions(precision=3, suppress=True)
CUTS, TAU = (0.15, 0.6), 0.25

def make_region(seed, U=60, M=400, flood_shift=0.0, bias_shift=0.0, road_effect=0.9):
    """A toy region: U units of M plots. True flooding varies by unit and plot. The map sees the truth plus noise,
    a per-unit bias, a global bias (bias_shift) and is fooled near roads."""
    r = np.random.default_rng(seed)
    unit = np.repeat(np.arange(U), M); road = r.exponential(1.0, U * M); near = np.exp(-road / 0.6)
    a = r.normal(-1.6 + flood_shift, 0.8, U)                                         # unit flood level
    y = (r.random(U * M) < expit(a[unit] + r.normal(0, 1.3, U * M) - road_effect * near)).astype(float)   # near-road plots flood less
    z = (2 * y - 1) * 1.3 + bias_shift + r.normal(0, 0.5, U)[unit] + r.normal(0, 1.0, U * M) + 0.9 * near   # map evidence: near-road looks wetter
    f = expit(z)                                                                       # calibrated-looking map probability
    return dict(U=U, unit=unit, road=road, y=y, f=f, h=np.searchsorted(CUTS, f, side="right"))

def prep(reg):
    U = reg["U"]; g = reg["unit"] * 3 + reg["h"]; n = np.bincount(reg["unit"])
    reg["g"] = g
    reg["W"] = np.bincount(g, minlength=U * 3).reshape(U, 3) / n[:, None]                              # W_uh
    cnt = np.bincount(g, minlength=U * 3)
    reg["Fb"] = np.where(cnt > 0, np.bincount(g, weights=reg["f"], minlength=U * 3) / np.maximum(cnt, 1), 0).reshape(U, 3)   # Fbar_uh
    reg["theta_map"] = np.bincount(reg["unit"], weights=reg["f"]) / n
    reg["theta_true"] = np.bincount(reg["unit"], weights=reg["y"]) / n
    return reg

train = prep(make_region(1, U=100))
print("strata sizes (share of training plots):", np.bincount(train["h"]) / len(train["h"]))
print("unit 0:  W_uh =", train["W"][0], " Fbar_uh =", train["Fb"][0], " map mean = %.3f, truth = %.3f" % (train["theta_map"][0], train["theta_true"][0]))
print("check: sum_h W_uh * Fbar_uh = %.3f  (equals the map mean)" % (train["W"][0] * train["Fb"][0]).sum())

# the rectifier e = y - f, averaged per stratum on the TRAINING region: this is 'Rpool'
e = train["y"] - train["f"]
Rpool = np.array([e[train["h"] == j].mean() for j in range(3)])
print("\nRpool (mean of truth minus map, per stratum):", Rpool)
print("negative = the map over-reads flooding in that stratum, most of all in the 'uncertain' stratum")
print("(Real run: Rpool = [-0.035, -0.207, -0.140], so the real map over-reads on training events as well.)")

**Say it to the guide (section 2):** "We aggregate Layer 2 pixels into 80 m plots and 320 m visit sites inside each unit. For each unit and stratum we store the area share W and the mean map probability Fbar. The three regions are split by event: seven training events with 125 units, two validation events with 27 units, two test events with 34 units. The field checks are simulated from the hand labels."

## 3. What is fitted on the training region (and only there)

All of these are fitted on **training events only** (`fit_hyper`) and then **frozen** before the validation or test region is touched.

| Quantity | Meaning | Real value |
|---|---|---|
| **Rpool_h** | the training-region average rectifier (truth minus map) in each stratum: the prior for a unit with no checks | **(-0.035, -0.207, -0.140)** |
| **T** | **between-unit** covariance of the stratum rectifiers around their own event's pool (how far one unit's map error can sit from its event's pool) | fitted |
| **T_ev** | **between-event** covariance of the event pools around Rpool (how far a **new event's** map error can sit from the training pool) | fitted over **7** events |
| **sigma2_h** | within-stratum plot variance of the rectifier | (0.0043, 0.0303, 0.0274) |
| **ICC** | intra-site correlation of the rectifier: how alike plots in the same site are | **0.33** |
| **deff** | design effect `1 + (k - 1) * ICC` for checking k neighbouring plots per visit | **2.0** (k = 4) |
| **m** | prior strength (see section 4), chosen by trying 0.5, 1, 2, 4, 8, 16, 32 on random check draws **on the training region** | **1.0** |
| **map_bias**, **map_rmse** | the raw map's mean error and RMSE on training units | **+0.080**, **0.128** |
| **pi_h** | the true flooded rate in each stratum (used by the older "report formula") | (0.009, 0.102, 0.768) |

Words:

* **Covariance matrix**: a table showing how the three strata's errors move together.
* **ICC (intraclass correlation)**: 0 means plots in a site are no more alike than plots anywhere; 1 means identical. ICC 0.33 means neighbours are quite alike.
* **Design effect (deff)**: the factor by which clustering inflates the variance compared with independent sampling. Checking 4 neighbouring plots with ICC 0.33 is worth only **4 / 2.0 = 2 independent plots**.

**What the negative Rpool means (be able to say it):** on the training events the map **over-reads** flooding, so the correction subtracts. But on a **new event** (India) the map may **under-read**. That is why T_ev, the between-event spread, exists. This lesson cost us a failed first version (section 7).

**Why choose m on the training region only:** m controls how strongly a unit is pulled toward the pool. Choosing it on validation or test would be tuning on events we are scoring. In the real run the m grid RMSE was nearly flat (0.0568 to 0.0692 over m from 0.5 to 32), minimum around **m = 1**.

## 4. The estimator, step by step (the heart of Layer 3)

Everything here is for **one unit u and one stratum h**. Then we add across strata.

### 4.1 Importance weights and the Hajek rectifier

Each checked plot c has a known inclusion probability **pi_c** and weight **d_c = 1 / pi_c**, and pixel count **w_c**. Define **a_c = d_c * w_c**. The **weighted average rectifier** in unit u, stratum h:

**R_uh = sum( a_c * e_c ) / sum( a_c )**   over checked plots c in (u, h)

This is a weighted mean, called the **Hajek** estimator (a "ratio" form: divide by the sum of the weights, not by a known total). The weights undo the unequal chances of selection (notebook 1, section 5).

### 4.2 How much do the checks count? Effective sample size and shrinkage

Weights that vary a lot make a sample worth fewer "real" checks. The **Kish effective sample size** is

**n_eff = (sum a)^2 / sum(a^2)**

(equal weights give n_eff = number of checks; very unequal weights give much less.)

The **shrinkage weight** is **lam_uh = n_eff / (n_eff + m)**. With no checks lam = 0; with many checks lam approaches 1. It says how much to trust this unit's own checks against the pooled prior.

### 4.2b The pooled prior P_h

When a unit has few checks we borrow strength from the **whole region's** checks. **v2** (current) builds P_h as a **precision-weighted mix** of the training pool **Rpool_h** (uncertain by **T_ev**, the new-event spread) and the **importance-weighted pooled rectifier of all checks in this region** (with a cluster-robust variance across units). **Precision** = 1 / variance; a precision-weighted mix trusts the better-known source more. Intuition: if the region's own checks say the map is badly off, that overrides the training pool.

### 4.3 The corrected rectifier and theta

**Rt_uh = lam_uh * R_uh + (1 - lam_uh) * P_h**   (shrink the unit's own rectifier toward the pooled prior)

**theta_u = sum over h of W_uh * ( Fbar_uh + Rt_uh )**

In words: for each stratum, take the map's average probability **Fbar** and **add the estimated correction Rt**, then average the strata by their area shares W. With no checks theta_u is the map plus the pooled correction. Final value is clipped to [0, 1].

### 4.4 The standard error s_u

**s_u^2 = V_sampling + V_shrink**

* **V_sampling** is the **cluster-robust variance** over visit sites (section 6), blended with a model variance when a unit has few sites.
* **V_shrink = a' T a + sum_h a_h^2 Var(P_h)**, with **a_h = W_uh * (1 - lam_uh)**. It is the uncertainty from the part of the answer that came from the pool rather than the unit's own checks: the bigger the pool's share (1 - lam), the bigger this term.

### 4.5 The comparison methods ("baselines beside every model")

| Method name in code | What it is |
|---|---|
| `map_only` | raw map mean; s = the training map RMSE (0.128). No checks used |
| `sample_only` | Hajek mean of the checked plots only; map ignored |
| `report_formula` | the older partial-report formula: `p_uh = (k_uh + m pi_h) / (n_uh + m)` with an **iid** (independent) variance and inclusion probabilities **ignored** (this is the formula Layer 5 uses for live updates; see notebook 5) |
| `ppi_v1` | the first version of our estimator: shrinks toward the training pool treated as **exact** |
| `ppi_naive` | our estimator with weights ignored (d_c = 1) |
| **`ppi_ipw`** | **our estimator** (Hajek weights, v2 pooling, cluster-robust variance) |
| `ppi_ipw_iid` | same estimate with the independent (non-clustered) variance, to show what clustering does to coverage |
| `ppi_ht` | the **unshrunk Horvitz-Thompson** form: the design-unbiased reference |

### 4.6 The Horvitz-Thompson (HT) form

In `ppi_ht` (with m near 0) the rectifier uses the **known total** of the stratum as its denominator: `R = sum(a e) / N_uh` instead of dividing by sum(a). It is **design-unbiased even if a unit's draw misses the low-probability sites**, whereas Hajek is only approximately unbiased. The price is higher variance (it is the "wrong calls" laggard in the tables). It is kept as the **bias reference** in the route stress test.

Try it: implement the simplified estimator on the toy region and compare with the baselines over many random check draws. (This is the idea, simplified: no clustering, no site structure, a fixed training pool.)

In [ ]:
def draw(reg, b, rng, design):
    """Poisson draw of plots; expected b checks per unit. 'route': plots near a road are far likelier to be picked.
    Returns the chosen plot indices and their KNOWN inclusion probabilities."""
    if b == 0: return np.zeros(0, int), np.zeros(0)
    a = np.ones(len(reg["unit"])) if design == "random" else 0.05 + np.exp(-reg["road"] / 0.6)
    asum = np.bincount(reg["unit"], weights=a, minlength=reg["U"])
    pi = np.clip(b * a / asum[reg["unit"]], 0, 1)
    sel = rng.random(len(pi)) < pi
    return np.flatnonzero(sel), pi[sel]

def est_ppi(reg, idx, pi, Rpool, m=1.0, weighted=True):
    U = reg["U"]; e = (reg["y"] - reg["f"])[idx]; g = reg["g"][idx]
    a = 1 / pi if weighted else np.ones(len(idx))                         # a_c = d_c * w_c (all plots weigh the same here)
    S = np.bincount(g, weights=a, minlength=U * 3); S2 = np.bincount(g, weights=a * a, minlength=U * 3)
    R = (np.bincount(g, weights=a * e, minlength=U * 3) / np.maximum(S, 1e-300)).reshape(U, 3)       # Hajek rectifier R_uh
    neff = np.where(S2 > 0, S * S / np.maximum(S2, 1e-300), 0).reshape(U, 3)                          # Kish effective size
    lam = neff / (neff + m)                                                                           # shrinkage weight
    Rt = lam * R + (1 - lam) * Rpool[None, :]                                                         # shrink toward the pool
    return np.clip((reg["W"] * (reg["Fb"] + Rt)).sum(1), 0, 1)                                        # theta_u

def est_sample_only(reg, idx, pi, prior):
    U = reg["U"]; u = reg["unit"][idx]; a = 1 / pi
    S = np.bincount(u, weights=a, minlength=U)
    R = np.bincount(u, weights=a * reg["y"][idx], minlength=U) / np.maximum(S, 1e-300)
    return np.where(S > 0, R, prior)

# NEW region = a new event: a different flood level, and a map that over-reads MORE than in training
new = prep(make_region(2, U=60, flood_shift=0.3, bias_shift=-0.6))
T = new["theta_true"]
print("new region: mean truth %.3f | mean raw map %.3f | share of units truly above 0.25: %.2f" % (T.mean(), new["theta_map"].mean(), (T > TAU).mean()))

def evaluate(design, budgets=(0, 4, 8, 16, 32, 64), reps=60):
    rows = []
    for b in budgets:
        rng = np.random.default_rng(10 + b); out = {"map_only": [], "sample_only": [], "ppi_naive": [], "ppi_ipw": []}
        for _ in range(reps):
            idx, pi = draw(new, b, rng, design)
            out["map_only"].append(new["theta_map"]); out["sample_only"].append(est_sample_only(new, idx, pi, train["y"].mean()))
            out["ppi_naive"].append(est_ppi(new, idx, pi, Rpool, weighted=False)); out["ppi_ipw"].append(est_ppi(new, idx, pi, Rpool))
        for k, v in out.items():
            v = np.array(v); err = v - T[None]
            rows.append(dict(budget=b, method=k, bias=err.mean(), rmse=np.sqrt((err ** 2).mean()), wrong=((v > TAU) != (T > TAU)[None]).mean()))
    return pd.DataFrame(rows)

res = evaluate("random")
print("\nRANDOM design (wrong = share of units on the wrong side of the 25 percent line)")
print(res.pivot(index="method", columns="budget", values="wrong").round(3).to_string())
print("\nRMSE"); print(res.pivot(index="method", columns="budget", values="rmse").round(3).to_string())
print("\nLook: the map-only error never improves (it is biased); checks-only is noisy at small budgets;")
print("the corrected map is best across the board. With 0 checks it is still the map plus the training correction.")

**Say it to the guide (section 4):** "For each unit and stratum the checks give a Hajek-weighted average of the map's error. It is shrunk toward a pooled prior by lam equal to the effective sample size over itself plus m. The unit's theta is the sum over strata of area share times the map's mean plus the corrected error. The pooled prior combines the training correction, with its between-event uncertainty, and the region's own checks. The standard error has a cluster-robust sampling part and a shrinkage part. We always compare with map-only, checks-only, the old report formula, the first version, an unweighted version and an unshrunk Horvitz-Thompson reference."

## 5. How checks are drawn, and the route-style stress test (sections 3 and 6)

### 5.1 The two-stage draw (`draw_checks`)

1. **Stage 1, visit sites:** each site is selected by **Poisson sampling** (each site independently with its own probability pi_s). The expected number of sites per unit is **budget / k** with k = 4.
2. **Stage 2, plots:** inside each selected site, pick **k = 4 plots** by simple random sampling without replacement.
3. **Inclusion probability of a plot: pi_c = pi_s * min(1, k / M_s)**, where M_s is the number of plots in the site. Exactly known.

### 5.2 Designs

* **random**: pi_s equal for all sites in a unit.
* **route**: `pi_s` proportional to `0.05 + exp(-road_km / 0.6)`: sites near a road are drawn many times more often than far sites. The probabilities are **still known and recorded**.
* **route + unreached** (`unreached = 0.5`): after drawing, far sites are **dropped** with probability up to 0.5 (the day ran out), **but the estimator is still given the nominal pi**. This tests what happens when the stated probabilities are wrong. It is the case flagged as the problem in the partial report, section 4.4: **nominal** (planned) versus **realised** inclusion probabilities.

### 5.3 Why route-style checking is dangerous

In our world (and plausibly in reality), **near-road plots are less flooded than the map thinks**. An officer who only checks near roads sees mostly dry plots, concludes "the map over-reads", and pulls every estimate **down**. That is selection bias. If you **weight each check by 1 / pi**, the near-road checks (high pi) get small weights and the rare far checks (low pi) get large weights, so the bias is largely removed.

Try it: the same toy region with route-style selection, weighted and unweighted.

In [ ]:
res_r = evaluate("route", budgets=(4, 8, 16, 32, 64))
print("ROUTE design: bias of theta_u (truth is the target; 0 is perfect)")
print(res_r.pivot(index="method", columns="budget", values="bias").round(3).to_string())
print("\nRMSE")
print(res_r.pivot(index="method", columns="budget", values="rmse").round(3).to_string())
print("\nThe unweighted version (ppi_naive) keeps a big negative bias even at 64 checks per unit: it only saw dry near-road plots.")
print("Weighting by 1/pi (ppi_ipw) cuts the bias to a fraction of that. The real run shows the same pattern:")
print("  real TEST, route design, 32 checks per unit: bias  naive -0.030 | Hajek -0.011 | HT +0.002")

**Say it to the guide (section 5):** "Checks are drawn in two stages: Poisson sampling of 320 m visit sites with an expected budget over four sites per unit, then four plots per selected site. The inclusion probability is the site probability times the plot fraction, and is known. Under a route-style design near-road sites are visited much more often, and since near-road plots flood less than the map says, an unweighted estimator is biased low. Weighting by one over pi removes most of that. We also test the case where far sites are dropped but the nominal probabilities are used."

## 6. Why clustering matters: cluster-robust variance (section 3, `variance.py`)

### 6.1 The problem

Plots in the same site are alike (flood together, mapped wrongly together). If we pretend four neighbouring plots are four independent checks, we **under-state the uncertainty**: our intervals are too narrow and the coverage drops below the nominal 90 percent.

### 6.2 The fix: the ultimate-cluster variance

The estimator is a smooth function of weighted sums, so by **linearisation** each checked plot c contributes an **influence value z_c** to the unit's estimate (the estimate moves by about `sum z_c`). Add the z values **within each visit site** to get site totals `t_s`. Then

**V_cr = n_s / (n_s - 1) * sum_s ( t_s - mean(t) )^2**

where n_s is the number of visit sites in the unit. In words: **treat each site as one observation** and measure how much the site totals vary. This does not need any assumption about how alike neighbouring plots are.

### 6.3 With very few sites: a moderated blend

With one or two sites V_cr is undefined or very noisy. So it is blended with a model variance fitted on the training region (`V_model` = pooled within-stratum variance x design effect) using `k0 = 2` pseudo degrees of freedom:

**V = ( k0 * V_model + (n_s - 1) * V_cr ) / ( k0 + n_s - 1 )**

So with n_s <= 1 it is V_model; as the number of sites grows it tends to V_cr.

Try it: show that the independent-plot standard error under-covers when plots are clustered, and the cluster-robust one does not.

In [ ]:
rng = np.random.default_rng(3)
n_sites, k, reps = 8, 4, 6000
mu, sd_site, sd_eps = 0.3, 0.12, 0.12                      # ICC = sd_site^2 / (sd_site^2 + sd_eps^2) = 0.5
icc = sd_site**2 / (sd_site**2 + sd_eps**2)
cover_iid = cover_cl = 0
se_iid_all, se_cl_all, means = [], [], []
for _ in range(reps):
    site_eff = rng.normal(0, sd_site, n_sites)
    y = mu + site_eff[:, None] + rng.normal(0, sd_eps, (n_sites, k))        # n_sites x k checked plots
    m = y.mean()
    se_iid = y.std(ddof=1) / np.sqrt(y.size)                                 # pretend all 32 plots are independent
    t = y.mean(1)                                                            # one number per site
    se_cl = t.std(ddof=1) / np.sqrt(n_sites)                                 # ultimate-cluster: sites are the unit of variation
    cover_iid += abs(m - mu) <= 1.645 * se_iid; cover_cl += abs(m - mu) <= 1.645 * se_cl
    se_iid_all.append(se_iid); se_cl_all.append(se_cl); means.append(m)
print(f"ICC = {icc:.2f}; design effect 1+(k-1)*ICC = {1 + (k - 1) * icc:.2f}")
print(f"true spread of the estimate (sd over repeats): {np.std(means):.4f}")
print(f"average iid standard error:        {np.mean(se_iid_all):.4f}   -> 90% interval covers the truth {cover_iid / reps:.2f} of the time")
print(f"average cluster-robust std error:  {np.mean(se_cl_all):.4f}   -> 90% interval covers the truth {cover_cl / reps:.2f} of the time")
print("The iid version is over-confident (too narrow). The real run shows the same: TEST, 16 checks, coverage ppi_ipw 0.806 vs ppi_ipw_iid 0.760.")

**Say it to the guide (section 6):** "Plots in the same 320 m site flood together and are mismapped together, with an intra-site correlation of 0.33 in our data, so a design effect of 2: four neighbouring plots are worth about two independent ones. We therefore compute the variance treating each visit site as one observation, the ultimate-cluster variance, and blend it with a model variance when a unit has very few sites. Pretending plots are independent gives intervals that are too narrow."

## 7. Version history: the honest record (section 3 of the notebook)

**Always say this in the viva, before the guide asks.**

* **v1** shrank every unit toward the **training pool** `Rpool_h` and treated that pool as **exact**. On its **first and only clean test run** (real Layer 2 maps, events Bolivia and India) v1 **missed its coverage bound**: the **India map under-reads** flooding while the training events' maps **over-read** it, so shrinking toward the training pool pulled every estimate down. Frozen record (`test_first_run_v1.json`): 90 percent interval **coverage 0.74 at 16 checks per unit**.
* **v2** (used now) adds the **between-event spread T_ev** and lets the pool be **learnt from the region's own checks** (section 4.2b).
* **v2 was written after seeing the v1 test result**, and was developed on the validation region only. **Therefore the v2 test numbers are NOT a clean held-out estimate.** The v1 numbers are kept in `test_first_run_v1.json`, printed in the test cell, and `ppi_v1` stays in every table for comparison.

This is what honest reporting looks like: the first test run failed, we said so, we fixed the method, and we label the later numbers as not clean.

## 8. Calls, P_u and the farmer-report flip value (section 4)

### 8.1 Calls (`make_calls`)

* `call = theta > tau` (above or below), `tau = 0.25`.
* `P_u = Phi((theta - tau) / s)` (the normal cumulative probability; see notebook 5, section 1).
* `P_right = P_u` if the call is above, `1 - P_u` if below.
* `margin_sd = (theta - tau) / s`: how many standard errors the estimate sits from the line.

### 8.2 Farmer reports are never mixed into theta_u

Farmers choose whether to report (**self-selected**), usually when they were flooded, so reports overstate flooding. The code simulates reports: a plot's farmer reports with probability **0.03 + 0.12 * y_c** (flooded farmers report more), says "flooded" when more than half the plot is flooded, plus **10 percent false "flooded" claims**. These are **synthetic**.

### 8.3 The flip value b*

Suppose a unit has **n_F reports** with mean claimed flooded share **rbar**. Suppose we decided to trust them as an estimate of the unit's share, with an unknown bias **b** (reports overstate flooding by b). The precision-weighted combination of our estimate and the reports is

**theta_c(b) = ( theta/s^2 + (rbar - b)/v_F ) / ( 1/s^2 + 1/v_F )**, with **v_F = max( rbar(1 - rbar), 0.05 ) / n_F**

The **flip value b\\*** is the bias at which theta_c(b\\*) lands exactly on tau. Solving: **b\\* = rbar - ( tau * (1/s^2 + 1/v_F) - theta/s^2 ) * v_F**.

How to read it. `call_c0` is the combined call if the reports were trusted at face value (b = 0). If it **equals** our call, the reports change nothing. If it **flips** our call, ask: how biased would the reports have to be for that flip to be wrong? **b_star is that bias**: the flip is justified only while the reports overstate flooding by **less than b_star**. A **large b_star** means the flip survives even quite biased reports (take it seriously and inspect). A **b_star near 0 or negative** means the flip needs nearly perfect reports, which self-selected reports are not (do not trust it).

Try it.

In [ ]:
def make_calls(theta, s, tau=0.25):
    theta, s = np.asarray(theta, float), np.maximum(np.asarray(s, float), 1e-9)
    call = theta > tau; P = ndtr((theta - tau) / s)
    return dict(call=call, P=P, P_right=np.where(call, P, 1 - P), margin_sd=(theta - tau) / s)

def flip_value(theta, s, rbar, n_F, tau=0.25):
    theta, s2, rbar, n_F = np.asarray(theta, float), np.maximum(np.asarray(s, float), 1e-9) ** 2, np.asarray(rbar, float), np.asarray(n_F, float)
    vF = np.maximum(rbar * (1 - rbar), 0.05) / n_F
    prec = 1 / s2 + 1 / vF
    theta_c0 = (theta / s2 + rbar / vF) / prec
    b_star = rbar - (tau * prec - theta / s2) * vF
    return dict(b_star=b_star, theta_c0=theta_c0, call_c0=theta_c0 > tau)

c = make_calls([0.10, 0.22, 0.27, 0.45], [0.05, 0.06, 0.06, 0.10])
print(pd.DataFrame({"theta": [0.10, 0.22, 0.27, 0.45], "s": [0.05, 0.06, 0.06, 0.10], "call": np.where(c["call"], "above", "below"),
                    "P_u": c["P"], "P_right": c["P_right"], "margin_sd": c["margin_sd"]}).round(3).to_string(index=False))

scenarios = {"A: reports say 0.90 flooded, we say 0.20": (0.20, 0.06, 0.90, 10),
             "B: reports say 0.45, we say 0.24 (borderline)": (0.24, 0.05, 0.45, 5),
             "C: reports say 0.35, we say 0.10": (0.10, 0.05, 0.35, 3)}
for name, (th, s, rb, nf) in scenarios.items():
    fv = flip_value(th, s, rb, nf)
    flipped = bool(fv["call_c0"]) != (th > 0.25)
    print(f"{name}\n   combined estimate if reports are trusted at face value: {float(fv['theta_c0']):.3f} -> call {'above' if fv['call_c0'] else 'below'}"
          f" | our call {'above' if th > 0.25 else 'below'} | flipped: {flipped} | b* = {float(fv['b_star']):+.3f}")
print("\nHow to read b*: it is the amount by which the reports could overstate flooding and still give a combined estimate ON the line.")
print("A: trusting the reports flips our call to 'above', and b* = +0.53 means that verdict survives even if the reports overstate by 53 points. Take it seriously: inspect.")
print("B: the flip only just happens and b* is about 0: it needs perfectly unbiased reports. Self-selected reports are not, so do not trust this flip.")
print("C: no flip at all, so the reports change nothing.")
# verify by plugging b* back in for scenario A
th, s, rb, nf = scenarios["A: reports say 0.90 flooded, we say 0.20"]
vF = max(rb * (1 - rb), 0.05) / nf; bs = float(flip_value(th, s, rb, nf)["b_star"])
print("\ncheck for A: combined estimate with bias b* removed = %.4f (should be 0.25)" % ((th / s**2 + (rb - bs) / vF) / (1 / s**2 + 1 / vF)))

**Say it to the guide (section 8):** "Every unit gets a call and P_u, the normal-curve probability that its true share is above 25 percent. Farmer reports are self-selected, so they are never mixed into theta. Instead we give a flip value: the bias in the reports at which trusting them would put the combined estimate exactly on the line. A large flip value means the call is robust to biased reports, a small one means the call depends on trusting them. The reports in the notebook are synthetic."

## 9. How the estimators are scored: Monte Carlo on events not used for fitting (sections 5 and 7)

### 9.1 The experiment

The **region is fixed**; only the **random check draw changes**, with fixed seeds. For each budget (0, 4, 8, 16, 32, 64 checks per unit on average) we draw checks **120 times** on the validation region and **200 times** on the test region, run **all methods** on the same draws, and compute metrics against the true unit shares. **Every method answers every unit.**

### 9.2 The metrics (`K.score`)

| Metric | Meaning | Good |
|---|---|---|
| **bias** | mean signed error | near 0 |
| **abs_unit_bias** | mean over units of the absolute average error (shrinkage can bias a single unit even when the overall average is fine) | low |
| **rmse** | root mean squared error of theta_u | low |
| **cover90** | share of 90 percent intervals (theta +/- 1.645 s) that contain the truth | near 0.90 |
| **wrong** | share of units whose call (above/below) is wrong | low |
| **brier** | mean squared error of P_u against "truly above" | low |
| **ece** | expected calibration error of P_u (10 bins): among units with P_u near 0.7, about 70 percent should truly be above | low |
| **s_over_rmse** | RMS of the stated s divided by the realised RMSE; 1 means honest | near 1 |

### 9.3 Reliability of P_u

Pool all draws and units at 16 checks per unit. Plot stated P_u (x) against the fraction truly above the line (y). A calibrated P_u lies on the diagonal. The ECE of that curve is reported.

## 10. The real results (from our committed run)

Source: `artifacts/layer3/validation.json` and `test.json`. Real Layer 2 maps; units are chip windows; **checks simulated from hand labels**. Columns are budgets (checks per unit).

### 10.1 Test region, random design: share of units with a WRONG call

| Method | 0 | 4 | 8 | 16 | 32 | 64 |
|---|---|---|---|---|---|---|
| `map_only` | 0.147 | 0.147 | 0.147 | 0.147 | 0.147 | 0.147 |
| `sample_only` (checks only) | 0.265 | 0.201 | 0.164 | 0.131 | 0.098 | 0.045 |
| `report_formula` | 0.206 | 0.160 | 0.129 | 0.096 | 0.064 | 0.034 |
| **`ppi_ipw` (ours)** | **0.206** | **0.139** | **0.120** | **0.098** | **0.068** | **0.038** |

### 10.2 Test region, random design: other metrics for `ppi_ipw`

| Metric | 0 | 4 | 8 | 16 | 32 | 64 |
|---|---|---|---|---|---|---|
| RMSE | 0.222 | 0.123 | 0.095 | 0.067 | 0.042 | 0.021 |
| 90% coverage | 0.647 | 0.725 | 0.755 | 0.806 | 0.870 | 0.963 |
| ECE of P_u | 0.193 | 0.076 | 0.060 | 0.041 | 0.022 | 0.019 |
| bias | -0.148 | -0.042 | -0.020 | -0.007 | -0.002 | 0.000 |

`map_only` has RMSE 0.160, bias -0.077, coverage 0.824, ECE 0.071 at every budget.

### 10.3 How to read the test table (what you must be able to say)

1. **Checks help a lot.** At 32 checks per unit our wrong-call rate is **0.068, against 0.147 for the map alone and 0.098 for checks alone**. That is the claim in the closing assertion (a).
2. **With zero checks, ours is WORSE than the raw map** (0.206 against 0.147; RMSE 0.222 against 0.160). On the new test events the training-region correction does not transfer. The notebook says so: that is why `unit_calls.csv` also stores the raw map share and its P (`theta_map_raw`, `P_map_raw`). **Our method starts to beat the map from about 4 checks per unit** (0.139 against 0.147).
3. **Ours beats checks-only at every budget**; the gap closes as checks grow (0.265 against 0.206 at zero checks, 0.045 against 0.038 at 64).
4. **Ours and `report_formula` are close on wrong calls** (0.098 against 0.096 at 16; 0.068 against 0.064 at 32). The advantage of the new estimator shows in **RMSE** (0.067 against 0.081 at 16), **bias under route selection**, and **calibration**, not in a big gap in wrong calls at random sampling. Say that honestly.
5. **Coverage is below the nominal 0.90** at small budgets (0.806 at 16, 0.870 at 32): the standard errors are somewhat **too small** on a new event. The independent-variance version is worse (0.760 at 16).
6. **P_u calibration**: ECE 0.041 at 16 checks, below the 0.06 bound, against 0.071 for the map alone.

### 10.4 Validation region, random design (for the val-test gap)

Wrong calls for `ppi_ipw`: **0.074, 0.061, 0.042, 0.021, 0.006, 0.000** for budgets 0 to 64 (`map_only` 0.074). Test is clearly harder (India has a map that under-reads). The **val-test gap in wrong calls at 16 checks is 0.098 minus 0.021 = about 0.077**. Say that gap out loud: the method degrades on new events, and the paper does not hide it.

### 10.5 Route stress test (test region)

| Wrong calls | 4 | 8 | 16 | 32 | 64 |
|---|---|---|---|---|---|
| `map_only` | 0.283 | 0.283 | 0.283 | 0.283 | 0.283 |
| `sample_only` | 0.224 | 0.187 | 0.146 | 0.118 | 0.087 |
| `ppi_naive` (weights ignored) | 0.138 | 0.123 | 0.107 | 0.093 | 0.067 |
| **`ppi_ipw`** | **0.123** | **0.111** | **0.094** | **0.077** | **0.062** |
| `ppi_ht` | 0.180 | 0.182 | 0.170 | 0.139 | 0.097 |

| Bias at 32 checks (route) | value |
|---|---|
| `ppi_naive` (unweighted) | **-0.030** |
| `ppi_ipw` (Hajek) | -0.011 |
| `ppi_ht` (unshrunk HT) | **+0.002** |
| with far sites **unreached** (nominal pi used), HT | -0.006 |

Reading: under route-style selection the **unweighted** estimator is biased by -0.030; **Hajek cuts the bias by about two thirds**; **HT is essentially unbiased** (+0.002) and stays close to zero even when far sites are not reached (-0.006), though HT is noisy (more wrong calls). Using the known inclusion probabilities matters.

### 10.6 The frozen first test run of v1 (budgets 0, 16, 32)

Wrong calls 0.206, 0.103, 0.069; coverage 0.588, 0.739, 0.822; ECE 0.196, 0.050, 0.017; RMSE 0.222, 0.082, 0.047 (`map_only`: 0.147, coverage 0.824, ECE 0.071, RMSE 0.160). v1 had the coverage problem described in section 7.

## 11. The closing assertions (section 10) and what they prove

Bounds stated **before** the test run. On the test region with random checks:

| Bound | Result in our run |
|---|---|
| (a) at **32** checks per unit the estimator makes **fewer wrong calls** than the map alone **and** than checks alone | **0.068** vs 0.147 and 0.098 |
| (b) its 90% interval **covers between 80% and 97%** at 16 and 32 checks | **0.806** and **0.870** |
| (c) **ECE of P_u at 16 checks is below 0.06** | **0.041** |
| (d) under **route** selection the **HT** form has |bias| below **0.005** at 32 checks, while the **unweighted** form is biased by **more than twice** that | HT +0.002, unweighted -0.030 |
| (e) **every unit is answered** | yes (34 of 34) |

Note (b): 0.806 is only just above the 0.80 floor. Do not present coverage as "close to 90 percent". It is "inside the bound we stated in advance".

**Between-region check:** the cell also runs the headline metrics at 16 checks per unit on **five more held-out synthetic regions** (new seeds, flood level shifts -0.3, 0, 0.15, 0.3, 0.5) to show how much results move from region to region. They are saved in `test.json` (`between_regions_b16`).

## 12. What Layer 3 writes (the contract with Layers 4 and 5)

| File | Content |
|---|---|
| `unit_calls.csv` | **today's state, before any field check** (n_checks = 0): `unit_id, region, theta, s, call, P_u, P_right, n_checks, n_sites, theta_map_raw, P_map_raw`; **34 test units**. Layers 4 and 5 start from here |
| `unit_calls_simulated_checks.csv` | the same after **one simulated draw of 16 checks per unit** (fixed seed), plus `farmer_reports, report_share, flip_b, theta_if_reports_trusted`. Labelled simulated |
| `unit_strata.npz` | per unit and stratum: `W_uh, Fbar_uh, n_uh, lam_uh, Rt_uh` (and `W_uh_n0`), for Layer 4's allocation |
| `hyper.json` | `tau`, strata cuts, `m`, `Rpool`, `T`, `sigma2`, `icc`, `deff`, `pi_h` (3 strata), `pi_h_2` (2 strata at 0.5, the "mapped dry / mapped flooded" split that Layer 4 uses), `map_rmse`, `m_grid_rmse`, `estimator_version = "v2 (event-level prior, region pooling)"` |
| `validation.json`, `test.json` | every metric table by method and budget; designs random, route, route-unreached |
| `reliability.json` | reliability curves of P_u at 16 checks |
| `test_first_run_v1.json` | the frozen v1 test result (not regenerated) |
| `MANIFEST.json` | schema, seed, input source, unit counts, hashes of the code and of the inputs, notes "Synthetic unless input says real. Checks ... are simulated. No abstain state." |

**Why `unit_calls.csv` includes the raw map columns:** with zero checks the training correction did not beat the raw map on either validation region, so we store both and let the tool show the raw map as a fallback. (This is the 'budget 0' column of the tables.)

## 13. The numbers to remember (mug this table)

| Thing | Value |
|---|---|
| Loss line **tau** | **0.25** |
| Call | `theta_u > 0.25`; always answered; no abstain |
| P_u | `Phi((theta_u - 0.25) / s_u)` |
| Strata cuts | **0.15** and **0.6** on the calibrated probability (fixed in advance): mapped dry / uncertain / mapped flooded |
| Plot | **4 x 4** pixels = **80 m**. Visit site | **16 x 16** pixels = **320 m**. Checks per visit **k = 4** |
| Budgets | **0, 4, 8, 16, 32, 64** checks per unit |
| Monte Carlo draws | **120** (validation), **200** (test) |
| Unit counts (real run) | train **125**, val **27**, test **34**; from **121** chip maps; at least **1,000** paddy px per unit |
| Training events | **7** (Mekong, Nigeria, Pakistan, Paraguay, Somalia, Spain, USA) |
| Rpool | **(-0.035, -0.207, -0.140)** |
| ICC / design effect | **0.33** / **2.0** |
| m | **1.0** (grid 0.5 to 32, chosen on training region) |
| Map bias / RMSE (training) | **+0.080** / **0.128** |
| Hajek weight | `d_c = 1 / pi_c`; `a_c = d_c * w_c` |
| Kish n_eff | `(sum a)^2 / sum a^2` |
| lam | `n_eff / (n_eff + m)` |
| theta | `sum_h W_uh (Fbar_uh + Rt_uh)`, `Rt = lam R + (1 - lam) P_h` |
| Inclusion probability | `pi_c = pi_s * min(1, k / M_s)` |
| Cluster-robust variance | `V_cr = n_s/(n_s-1) * sum_s (t_s - mean t)^2` |
| Moderated variance | `(k0 V_model + (n_s - 1) V_cr) / (k0 + n_s - 1)`, `k0 = 2` |
| Route weight | `pi_s ~ 0.05 + exp(-road_km / 0.6)` |
| Unreached | drop far sites with probability up to **0.5**, nominal pi kept |
| Farmer reports | report with probability **0.03 + 0.12 y**; **10 %** false claims; `v_F = max(rbar(1-rbar), 0.05)/n_F` |
| Flip value | `b* = rbar - (tau * (1/s^2 + 1/v_F) - theta/s^2) * v_F` |
| Test, 32 checks, wrong calls | ours **0.068**, map-only **0.147**, checks-only **0.098** |
| Test, 0 checks, wrong calls | ours **0.206**, map-only **0.147** (ours worse) |
| Test, 16 checks | coverage **0.806**, ECE **0.041**, RMSE **0.067** |
| Val-test gap (wrong calls, 16 checks) | 0.021 on validation vs 0.098 on test |
| Route bias at 32 checks | naive **-0.030**, Hajek **-0.011**, HT **+0.002** |
| v1 first test run | coverage **0.74** at 16 checks (missed its bound) |
| Estimator version | **v2** (event-level prior, region pooling), written after seeing v1's test result |
| Assertion bounds | wrong calls beat map and checks at 32; coverage in [0.80, 0.97] at 16 and 32; ECE at 16 below 0.06; HT route |bias| below 0.005 and naive over twice that; every unit answered |
| Seed | **20261118** |

## 14. Glossary (alphabetical)

* **Bias**: average error over repeats. Zero means unbiased.
* **Budget**: average field checks per unit.
* **Cluster-robust variance**: variance computed treating each visit site as one observation.
* **Coverage (cover90)**: fraction of 90 percent intervals containing the truth.
* **Covariance matrix**: table of how quantities vary together.
* **Design effect (deff)**: variance inflation due to clustering, `1 + (k-1) ICC`.
* **Effective sample size (Kish)**: `(sum a)^2 / sum a^2`; the number of equal-weight checks the sample is worth.
* **Estimator**: a recipe producing an estimate from data.
* **Frame**: the list of all plots that could be checked.
* **Hajek estimator**: weighted mean, weights summing in the denominator.
* **Horvitz-Thompson (HT)**: sum of y over pi, divided by the known total; design-unbiased.
* **ICC**: intraclass correlation, how alike plots in a site are.
* **Inclusion probability (pi)**: known chance a plot is checked.
* **Linearisation / influence value**: approximating an estimator by a sum of per-check contributions.
* **Monte Carlo**: repeating a random experiment many times by computer.
* **PPI (prediction-powered inference)**: use a prediction everywhere and a few real measurements to correct its error.
* **Precision**: 1 / variance.
* **Pooled prior (P_h)**: shared estimate of the stratum rectifier that units shrink toward.
* **Rectifier**: the map's error, truth minus map probability.
* **RMSE**: root mean squared error.
* **Route design**: check selection that favours places near roads.
* **Shrinkage**: pulling a noisy estimate toward a prior.
* **Standard error (s)**: the estimated typical size of an estimate's error.
* **Stratum**: group of plots by map score (dry, uncertain, flooded).
* **theta_u**: estimated flooded share of unit u's paddy area.
* **Visit site**: cluster of nearby plots checked from one stop.
* **Weight (1/pi)**: how many plots a checked plot stands for.

## 15. Questions an examiner may ask, and the one-line answers

1. **What does Layer 3 do?** Turns the calibrated map plus a few field checks into a flooded share theta_u, a standard error, an always-answered call and P_u for every unit.
2. **Why not just use the map?** It can be biased, especially on a new flood event. On our test events map-only gets 14.7 percent of calls wrong.
3. **Why not just use the checks?** They are too few and noisy. At 32 checks per unit checks-only gets 9.8 percent wrong against 6.8 percent for ours.
4. **What is prediction-powered inference?** Use the map as a prediction for every plot and use the checks only to estimate the map's average error, then correct it.
5. **What is the rectifier?** The map's error, truth minus map probability, averaged per stratum.
6. **Why does it work with few checks?** The map's error is a smaller, smoother quantity than the flooded share itself, so it is estimated more precisely from few checks.
7. **What are the strata?** Mapped dry (below 0.15), uncertain (0.15 to 0.6), mapped flooded (0.6 and above), fixed in advance.
8. **What is fitted on the training region?** Rpool, the between-unit and between-event covariances, within-stratum variance, ICC and the prior strength m.
9. **Why weight checks by 1 over pi?** To undo unequal selection chances, which route-style visiting creates. It makes the estimate approximately (Hajek) or exactly (HT) design-unbiased.
10. **What is shrinkage and what does m do?** Pulling a unit's own estimate toward a pooled prior, with weight lam equal to effective size over effective size plus m.
11. **What is cluster-robust variance and why?** Plots in a visit site are alike, so treating them as independent understates uncertainty. We treat each site as one observation.
12. **What is the design effect?** The variance inflation from clustering, 1 plus (k minus 1) times ICC. Ours is 2.0.
13. **What is P_u?** The probability the unit's true share exceeds 25 percent, Phi of (theta minus tau) over s.
14. **Why no abstain state?** Every unit needs a decision, and P_u already says how far to trust it. A weak call shows as P_u near 0.5.
15. **What is the flip value?** The bias in farmer reports at which trusting them would put the combined estimate exactly on the line. Reports are never mixed into theta.
16. **How did you evaluate?** Monte Carlo over many random check draws on events not used for fitting, with map-only and checks-only baselines, scoring bias, RMSE, coverage, wrong calls and calibration of P_u.
17. **How did you do on test?** At 32 checks per unit, 6.8 percent wrong calls against 14.7 percent for the map and 9.8 percent for checks alone; ECE of P_u 0.041 at 16 checks.
18. **What are the weaknesses?** With zero checks it is worse than the map; coverage is below 90 percent at small budgets (0.81 at 16); test is much harder than validation; the estimate is close to the old report formula on wrong calls.
19. **What happened with the first version?** It missed its coverage bound on the first test run (0.74) because India's map under-reads while training maps over-read. v2 adds the between-event spread and learns the pool from the region's own checks. v2 was written after seeing that result, so its test numbers are not a clean held-out estimate.
20. **What did the route stress test show?** Unweighted route-style checks are biased low (-0.030 at 32 checks); weighting by 1 over pi removes most of it; HT is nearly unbiased (+0.002) even with far sites unreached.
21. **Are the field checks real?** No. They are simulated from the Sen1Floods11 hand labels. No officer was observed.
22. **Which numbers go to Layers 4 and 5?** `unit_calls.csv` (today's state, no checks), `unit_strata.npz` and `hyper.json`.